# **LIbs**

In [15]:
import cv2 as cv
import numpy as np
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense,Flatten,Conv2D,MaxPooling2D,Dropout,GlobalAveragePooling2D
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from keras.models import load_model
from keras.applications.resnet_v2 import ResNet101V2
from keras.models import Model

# **Collecting Data**

In [5]:
import cv2
import numpy as np
import os


gesture_data = []
labels = []
gesture_label = None  


gesture_mapping = {
    's': 0,  
    'f': 1,  
    'o': 2  
}

# Set up video capture
cam = cv2.VideoCapture(0)

print("Press 's', 'f', or 'o' to assign labels to gestures.")
print("Press 'q' to quit and save the collected data.")

while True:
    ret, frame = cam.read()
    if not ret:
        break

    # Draw ROI (Region of Interest) on the frame
    cv2.rectangle(frame, (100, 100), (400, 400), (0, 255, 0), 2)
    roi = frame[100:400, 100:400]  # Crop the ROI for gesture

    # Preprocess ROI
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY) 
    resized = cv2.resize(gray, (64, 64))         
    normalized = resized / 255.0                 

    # Display the current frame
    cv2.imshow("Frame", frame)

    # Handle key presses for labeling gestures
    key = cv2.waitKey(1) & 0xFF
    if key in [ord('s'), ord('f'), ord('o')]:  
        gesture_label = gesture_mapping[chr(key)] 
        gesture_data.append(normalized)           
        labels.append(gesture_label)             
        print(f"Collected data for label: {gesture_label}")
    elif key == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()

# Convert data to numpy arrays
gesture_data = np.array(gesture_data).reshape(-1, 64, 64, 1)  # Add channel dimension
labels = np.array(labels)

print("Data collection complete.")
print(f"Gesture data shape: {gesture_data.shape}")
print(f"Labels shape: {labels.shape}")

# Save the collected data
os.makedirs("gesture_dataset", exist_ok=True)
np.save("gesture_dataset/gesture_data.npy", gesture_data)
np.save("gesture_dataset/labels.npy", labels)

print("Data saved successfully!")



Press 's', 'f', or 'o' to assign labels to gestures.
Press 'q' to quit and save the collected data.
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Collected data for label: 0
Coll

In [35]:
num_classes

3

# **Split_data_And_perform_one_hot_encoding_to_ensure_labels_for_integer_encoded**


In [36]:
# Load data
gesture_data = np.load("gesture_dataset/gesture_data.npy")
labels = np.load("gesture_dataset/labels.npy")

# Normalize data (already normalized during collection, ensure it's in range [0, 1])
gesture_data = gesture_data.astype('float32')

# One-hot encode labels (adjust num_classes based on the number of gestures)
num_classes = len(np.unique(labels))
labels = to_categorical(labels, num_classes=num_classes)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(gesture_data, labels, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

Training data shape: (335, 64, 64, 1)
Testing data shape: (84, 64, 64, 1)


# **model**

In [37]:
model = Sequential()

model.add(Conv2D(filters = 32, kernel_size = (3,3), input_shape = (64,64,1), activation = 'relu'))
model.add(MaxPooling2D(2,2))
model.add(Dropout(0.1))

model.add(Conv2D(filters = 64, kernel_size = (3,3),  activation = 'relu'))
model.add(MaxPooling2D(2,2))
model.add(Dropout(0.1))

model.add(Flatten())

model.add(Dense(units = 128, activation='relu'))
model.add(Dense(units = num_classes , activation='softmax'))


    


# **Compiling_model**

In [38]:
model.compile(optimizer="adam",loss='categorical_crossentropy',metrics=['accuracy'])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_2 (Conv2D)           (None, 62, 62, 32)        320       
                                                                 
 max_pooling2d_11 (MaxPoolin  (None, 31, 31, 32)       0         
 g2D)                                                            
                                                                 
 dropout_2 (Dropout)         (None, 31, 31, 32)        0         
                                                                 
 conv2d_3 (Conv2D)           (None, 29, 29, 64)        18496     
                                                                 


 max_pooling2d_12 (MaxPoolin  (None, 14, 14, 64)       0         
 g2D)                                                            
                                                                 
 dropout_3 (Dropout)         (None, 14, 14, 64)        0         
                                                                 
 flatten_1 (Flatten)         (None, 12544)             0         
                                                                 
 dense_5 (Dense)             (None, 128)               1605760   
                                                                 
 dense_6 (Dense)             (None, 3)                 387       
                                                                 
Total params: 1,624,963
Trainable params: 1,624,963
Non-trainable params: 0
_________________________________________________________________


# **Train**

In [39]:
history = model.fit(X_train,y_train,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
11/11 [==============================] - 1s 71ms/step - loss: 1.1418 - accuracy: 0.3612 - val_loss: 1.0307 - val_accuracy: 0.7143
Epoch 2/10
11/11 [==============================] - 1s 66ms/step - loss: 0.9431 - accuracy: 0.6209 - val_loss: 0.8475 - val_accuracy: 0.6667
Epoch 3/10
11/11 [==============================] - 1s 65ms/step - loss: 0.5935 - accuracy: 0.8657 - val_loss: 0.3864 - val_accuracy: 0.8929
Epoch 4/10
11/11 [==============================] - 1s 66ms/step - loss: 0.2508 - accuracy: 0.9463 - val_loss: 0.2259 - val_accuracy: 0.9048
Epoch 5/10
11/11 [==============================] - 1s 64ms/step - loss: 0.1659 - accuracy: 0.9582 - val_loss: 0.2036 - val_accuracy: 0.9048
Epoch 6/10
11/11 [==============================] - 1s 65ms/step - loss: 0.1295 - accuracy: 0.9582 - val_loss: 0.1276 - val_accuracy: 0.9524
Epoch 7/10
11/11 [==============================] - 1s 73ms/step - loss: 0.0854 - accuracy: 0.9821 - val_loss: 0.1159 - val_accuracy: 0.9643
Epoch 8/10
11

# **Save_model**

In [40]:
model.save(filepath='hand_gestures.h5')

In [41]:
# Load the trained model
model = load_model("hand_gestures.h5")

# Predict using a single image (example)
sample_image = X_test[-1].reshape(1, 64, 64, 1)  # Reshape for batch size
predicted_label = np.argmax(model.predict(sample_image))
print(f"Predicted Label: {predicted_label}")

Predicted Label: 0


# **test**

In [43]:
import cv2 as cv
import numpy as np
from tensorflow.keras.models import load_model
from collections import deque

# Load trained model
model = load_model("hand_gestures.h5")

# Gesture label mapping
gesture_mapping = {
    0: "Stop",
    1: "Fist",
    2: "Open Hand",
    
}

# Initialize video capture
cap = cv.VideoCapture(0)

# Store recent predictions for smoothing
predictions_buffer = deque(maxlen=10)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Flip frame horizontally for a mirror effect
    frame = cv.flip(frame, 1)

    # Draw ROI (Region of Interest)
    cv.rectangle(frame, (100, 100), (400, 400), (0, 255, 0), 2)
    roi = frame[100:400, 100:400]

   
    gray = cv.cvtColor(roi, cv.COLOR_BGR2GRAY)
    gray = cv.equalizeHist(gray)  
    resized = cv.resize(gray, (64, 64))
    normalized = resized / 255.0
    reshaped = np.reshape(normalized, (1, 64, 64, 1))

    # Predict gesture
    prediction = model.predict(reshaped, verbose=0)
    gesture_index = np.argmax(prediction)
    predictions_buffer.append(gesture_index)

    
    gesture = max(set(predictions_buffer), key=predictions_buffer.count)

    # Display gesture name on the frame
    gesture_name = gesture_mapping.get(gesture, "Unknown")
    cv.putText(frame, f"Gesture: {gesture_name}", (50, 50), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    # Show the frame
    cv.imshow("Hand Gesture Recognition", frame)


    if cv.waitKey(1) & 0xFF == ord('q'):
        break


cap.release()
cv.destroyAllWindows()